# 🎙 Stimmenkloner – Sprachboard

Dieses Notebook klont eine Stimme und generiert 8 Sätze als MP3-Dateien.

## ⚡ Wichtig vor dem Start
GPU aktivieren: **Laufzeit → Laufzeittyp ändern → T4 GPU** → Speichern

## 📋 Ablauf
1. Sprachprobe hochladen
2. Installation (einmalig, ~2 Min)
3. Modell laden (~3 Min)
4. Alle 8 Sätze generieren
5. ZIP mit allen MP3s herunterladen

---
## Schritt 1 – Sprachprobe hochladen

Lade eine Aufnahme der Stimme hoch, die du klonen möchtest.

**Anforderungen:**
- Mindestens **10–30 Sekunden** reine Sprache
- Format: **MP3 oder WAV**
- Möglichst kein Hintergrundgeräusch / keine Musik
- Eine normale Handy-Aufnahme reicht völlig

In [ ]:
from google.colab import files

print('Datei auswählen...')
uploaded = files.upload()

voice_file = list(uploaded.keys())[0]
print(f'✓ Hochgeladen: {voice_file}')

---
## Schritt 2 – Installation
Dauert ca. 1–2 Minuten. Nur beim ersten Mal nötig.

In [ ]:
!pip install -q TTS
!apt-get install -q ffmpeg
print('✓ Installation abgeschlossen')

---
## Schritt 3 – Modell laden
Das XTTS v2 Modell wird heruntergeladen (~2 GB). Dauert ca. 3 Minuten.
Beim ersten Mal fragt es nach Zustimmung – einfach `y` eingeben wenn gefragt.

In [ ]:
import torch
from TTS.api import TTS
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Verwende: {device.upper()}')

# Nutzungsbedingungen automatisch akzeptieren
os.environ['COQUI_TOS_AGREED'] = '1'

tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
print('✓ Modell geladen – bereit zur Generierung')

---
## Schritt 4 – Sätze generieren

Die 8 Sätze für das Sprachboard. Du kannst sie hier beliebig ändern.

In [ ]:
saetze = [
    "Wer auf Toilette möchte, hebt bitte die Hand.",
    "Heute geht die erste Runde Bier selbstverständlich auf mich.",
    "Der Herr ist mein Hirte. Mein Fahrer ist heute Pascal.",
    "Ich erkenne ein gutes Auto daran, wie bequem der Beifahrersitz ist.",
    "Mein Lieblingsauto ist das, in dem mich andere mitnehmen.",
    "Alkoholische Mitarbeit ist heute ausdrücklich erwünscht.",
    "Ich fühle mich wie 2009 im Bierkönig.",
    "Mein Verantwortungsbereich endet ab dem zweiten Bier.",
]

print(f'Generiere {len(saetze)} Sätze in der geklonten Stimme...\n')

for i, text in enumerate(saetze, 1):
    wav_path = f'/content/clip_{i:02d}.wav'
    mp3_path = f'/content/clip_{i:02d}.mp3'

    tts.tts_to_file(
        text=text,
        speaker_wav=voice_file,
        language='de',
        file_path=wav_path
    )

    # WAV → MP3
    os.system(f'ffmpeg -i {wav_path} -q:a 2 {mp3_path} -y -loglevel quiet')
    os.remove(wav_path)

    print(f'  ✓ Clip {i:02d}: {text[:55]}...' if len(text) > 55 else f'  ✓ Clip {i:02d}: {text}')

print('\n✓ Alle Clips fertig!')

---
## Schritt 5 – Herunterladen
Alle 8 MP3s als ZIP-Datei herunterladen.

In [ ]:
!zip -j /content/sprachboard_clips.zip /content/clip_*.mp3

from google.colab import files
files.download('/content/sprachboard_clips.zip')
print('✓ Download gestartet: sprachboard_clips.zip')

---
## Nächste Schritte

1. ZIP entpacken → 8 MP3-Dateien (`clip_01.mp3` bis `clip_08.mp3`)
2. Dateien in den Ordner `audio/` des Repos hochladen
3. Fertig – die Buttons auf der Website spielen die Clips ab